<div style="background: linear-gradient(135deg, #212529 0%, #343a40 100%); padding: 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: white; margin: 0; font-size: 2em;">🔬 Sandbox de Exploração - PCA</h1>
  <p style="color: #d1f2eb; margin: 8px 0 0 0; font-size: 1.1em;">Fase 01 · Projeto Final</p>
</div>

## 🎯 Projeto Final: PCA do Zero com NumPy

Este projeto sintetiza **toda a Fase 01**: álgebra linear (covariância, autovetores) + estatística (variância). Você vai implementar PCA sem sklearn.

**Regra de ouro:** Implemente primeiro com NumPy puro. Depois valide com sklearn.

---

## Etapa 1: Carregar e Explorar os Dados

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris

# Carregar Iris (4 features, 3 classes)
iris = load_iris()
X = iris.data           # (150, 4)
y = iris.target         # (150,) → 0, 1, 2
nomes_features = iris.feature_names
nomes_classes = iris.target_names

print(f'Shape: {X.shape}')
print(f'Features: {nomes_features}')
print(f'Classes: {nomes_classes}')
print(f'\nEstatísticas:')
for i, nome in enumerate(nomes_features):
    print(f'  {nome}: média={X[:, i].mean():.2f}, std={X[:, i].std():.2f}')

## Etapa 2: Centralizar os Dados (Subtrair a Média)

In [ ]:
# Centralizar
media = X.mean(axis=0)
X_c = X - media

print(f'Média original: {media.round(4)}')
print(f'Média após centralizar: {X_c.mean(axis=0).round(10)}')
print('→ Deve ser praticamente zero!')

## Etapa 3: Calcular a Matriz de Covariância

In [ ]:
# Fórmula: C = (X^T · X) / (n - 1)
n = X_c.shape[0]
C = (X_c.T @ X_c) / (n - 1)  # shape (4, 4)

# Validar com NumPy
C_np = np.cov(X_c.T)

print('Matriz de Covariância (manual):')
print(np.round(C, 4))
print(f'\nIgual ao np.cov? {np.allclose(C, C_np)}')

## Etapa 4: Autovalores e Autovetores

In [ ]:
autovalores, autovetores = np.linalg.eig(C)

# Ordenar por autovalor decrescente
ordem = np.argsort(autovalores)[::-1]
autovalores = autovalores[ordem]
autovetores = autovetores[:, ordem]

# Variância explicada
var_explicada = autovalores / autovalores.sum() * 100
var_acumulada = np.cumsum(var_explicada)

print('Componente | Autovalor | Var. Explicada | Var. Acumulada')
print('-' * 60)
for i in range(4):
    print(f'    PC{i+1}    | {autovalores[i]:>9.4f} | '
          f'{var_explicada[i]:>12.2f}% | {var_acumulada[i]:>12.2f}%')

## Etapa 5: Scree Plot (Gráfico de Variância Explicada)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barras
axes[0].bar(range(1, 5), var_explicada, color='steelblue', alpha=0.8)
axes[0].set_xlabel('Componente Principal')
axes[0].set_ylabel('Variância Explicada (%)')
axes[0].set_title('Scree Plot', fontsize=13, fontweight='bold')
axes[0].set_xticks(range(1, 5))

# Acumulado
axes[1].plot(range(1, 5), var_acumulada, 'ro-', linewidth=2, markersize=8)
axes[1].axhline(95, color='green', linestyle='--', label='95% threshold')
axes[1].set_xlabel('Número de Componentes')
axes[1].set_ylabel('Variância Acumulada (%)')
axes[1].set_title('Variância Acumulada', fontsize=13, fontweight='bold')
axes[1].set_xticks(range(1, 5))
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.show()

# Quantos PCs para 95%?
k_95 = np.argmax(var_acumulada >= 95) + 1
print(f'\n→ {k_95} componentes explicam ≥95% da variância!')

## Etapa 6: Projetar nos 2 Primeiros PCs

In [ ]:
# Projetar
k = 2
W = autovetores[:, :k]    # (4, 2)
X_pca = X_c @ W           # (150, 2)

# Visualizar
fig, ax = plt.subplots(figsize=(10, 8))
cores = ['#e74c3c', '#2ecc71', '#3498db']
for i, (classe, cor) in enumerate(zip(nomes_classes, cores)):
    mask = y == i
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], 
               c=cor, label=classe, s=50, alpha=0.7, edgecolors='white')

ax.set_xlabel(f'PC1 ({var_explicada[0]:.1f}%)', fontsize=13)
ax.set_ylabel(f'PC2 ({var_explicada[1]:.1f}%)', fontsize=13)
ax.set_title('Iris — PCA do Zero (2 componentes)', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.show()

## Etapa 7: Validar com sklearn

In [ ]:
from sklearn.decomposition import PCA

pca_sk = PCA(n_components=4)
X_pca_sk = pca_sk.fit_transform(X_c)

print('=== Validação ===')
print(f'Var. explicada (nosso):  {var_explicada.round(2)}')
print(f'Var. explicada (sklearn): {(pca_sk.explained_variance_ratio_ * 100).round(2)}')
print(f'\nProjeção igual (ou sinal invertido)?')
print(f'  Correlação PC1: {np.corrcoef(X_pca[:, 0], X_pca_sk[:, 0])[0, 1]:.6f}')
print(f'  Correlação PC2: {np.corrcoef(X_pca[:, 1], X_pca_sk[:, 1])[0, 1]:.6f}')
print('\n✅ Correlação ≈ ±1.0 confirma que nossa implementação está correta!')

## 🏋️ Questões para Praticar

### Q1. Erro de Reconstrução
Reconstrua os dados de 4D usando apenas 2 PCs: `X_rec = X_pca @ W.T + media`. Calcule o MSE entre original e reconstruído.

### Q2. PCA em MNIST
Carregue `sklearn.datasets.load_digits()` (64 features). Reduza a 2D e plote. Quantas PCs capturam 90% da variância?

### Q3. Biplot
Faça um **biplot**: dados projetados em 2D + setas mostrando a contribuição de cada feature original.

In [ ]:
# ============================================
# Resolva as questões aqui
# ============================================

# Q1: Erro de reconstrução


# Q2: PCA em MNIST


# Q3: Biplot
